MDP: Frozen Lake
================

**Author:** Joseph Le Roux

**Date:** 2026-07-19



- AUTHOR: Joseph Le Roux
- DATE: 2026-07-19
- DESCRIPTION: MDP Frozen Lake


## Understanding the Problem



### Exercise 1



In this exercise, you are asked to provide the probabilities of transitions $T(s,a,s')= P_{{\delta} (S_{t+1}=s' | S_t=s, A_t=a)$.
The MDP dynamics are described in the lab assignment.

-   The first student in the pair must find:
    
    Based on the MDP definition given in the tutorial, provide the following values for the transition function:
    
    -   $\forall s'\ T(0,0,s')$ , i.e., all transition values from the initial state when performing action 0 (left) for all states (only list non-zero transition values)
    
    -   $\forall s'\ T(9,a,s')$ , i.e., all transition values from state 9 when performing action $a$ for all states (only list non-zero transition values)
    
    -   $\forall s'\ T(62,a,s')$ , i.e., all transition values from state 62 when performing action $a$ for all states (only list non-zero transition values)

-   The second student in the pair must find:
    
    Based on the MDP definition given in the tutorial, provide the following values for the transition function:
    
    -   $\forall s'\ T(7,0,s')$ , i.e., all transition values from the top-right state when performing action 0 (left) for all states (only list non-zero transition values)
    
    -   $\forall s'\ T(11,a,s')$ , i.e., all transition values from state 11 when performing action $a$ for all states (only list non-zero transition values)
    
    -   $\forall s'\ T(55,a,s')$ , i.e., all transition values from state 55 when performing action $a$ for all states (only list non-zero transition values)



### Answers



**Write your answers here** (you can verify your answers in the dynamics question later in the assignment)



## Let Me Code Now!



In [1]:
# gym contains the RL problem interface through environment
import gymnasium as gym
# for vector/matrix/tensor data
import torch as t

import random

We create an environment object using the `gym.make` function



In [1]:
# load the frozen lake problem
env = gym.make('FrozenLake-v1', map_name='8x8', render_mode='ansi')

On the environment object, we can perform several operations. Here we initialize it (via reset) and display it (via render).

Note that the current position is indicated by a different color



In [1]:
env.reset()
print(env) #the environment is wrapped to add more features
print(env.unwrapped) # access the real (underlying) environment
print(env.render())

The `env` structure has two attributes to define states and actions:

1.  `observation_space`
2.  `action_space`



In [1]:
print(type(env.action_space))
print(env.action_space.n)

print(type(env.observation_space))
print(env.observation_space.n)

We can choose an action randomly using the **sample** function. To perform an action, we call **step**. In the following example, we randomly choose an action and apply it to the current state of the environment.

NB: the **step** function returns a tuple $(o,r,f,h,i)$ where $o$ is the new state, $r$ is the reward for reaching $o$, $f$ is a boolean indicating if $o$ is terminal, $h$ is a boolean indicating the horizon has been reached, and $i$ is debug information (which we will ignore for these labs).

Run the following block several times and try to understand what happens:



In [1]:
action = env.action_space.sample()
print(action)
res = env.step(action)
print("result:", res)
print(env.render())

### Dynamics and Rewards



In this environment, the dynamics are known and stored in a specific data structure `TableTransition`.

A `TableTransition` is a table `State` $\to$ `TableState` that maps each of the 64 states of the environment to the corresponding outgoing transitions table.

A `TableState` is a table `Action` $\to$ `List(Arrival)` that associates to each possible action the displacements that can be performed.

We call `Arrival` tuples of the form $(p,d,r,f)$ where $p$ is a probability, $d$ is an arrival state, $r$ is a reward for reaching $d$, and $f$ is a boolean indicating if state $d$ is terminal.



In [1]:
print(env.unwrapped.P)

### Exercise 2



From the structure displayed above, copy the `TableStates` for the transitions in Exercise 1



In [1]:
#your answer here

## Finding the Optimal Policy: Policy Iteration



### Computing the Optimal Policy



We start by defining an **evaluate** function that takes as input an environment **env**, a policy **policy**, and a number of trials to perform **trials**. It computes **trials** trajectories from the MDP dynamics and the **policy** to choose the action to perform at each time **t**.

If the **policy** argument is **None**, actions are chosen randomly.

**evaluate** computes the average success rate and average trajectory length for trajectories that end in G.



In [1]:
#set "render" to "True" to display intermediate boards

def evaluate(env, policy = None, trials=1000, render=False):

  success = 0
  lengths = []

  for trial in range(trials):
    obs,_ = env.reset()

    terminated = False
    truncated = False

    if render:
      env.render()
    length = 0

    while not (terminated or truncated):
      action = env.action_space.sample() if policy is None else policy[obs]
      obs,reward, terminated, truncated, _ = env.step(action)
      if render:
        print(env.render())
      length +=1

      if terminated:
        if reward == 1:
          success +=1
          lengths.append(length)

  print('----------------------------------------------')
  print('You retrieved the frisbee {:.2f}% of the time'.format((success/trials) * 100))
  print('On average it takes you {:.0f} moves to reach the frisbee'.format(t.mean(t.tensor(lengths, dtype=t.float))))
  print('----------------------------------------------')

In [1]:
env = gym.make('FrozenLake-v1', map_name='8x8', render_mode='ansi')
evaluate(env)

### Improving the Action Policy



In this section, we will implement the algorithms seen in class: **policy iteration** and **value iteration**.

In both cases, this implementation will take the form of a Python class with a `get_policy` method that takes a state number and returns the action to follow according to a deterministic policy.



### Policy Iteration Algorithm



In [1]:
class PolicyIteration:
  def __init__(self, env):

    self.pi = t.zeros(env.observation_space.n, dtype=t.long)
    self.values = t.zeros(env.observation_space.n)

    # transition T(s,a,s')
    self.transitions = t.zeros((env.observation_space.n,env.action_space.n,env.observation_space.n))

    # reward R(s,a,s')
    self.rewards = t.zeros((env.observation_space.n,env.action_space.n,env.observation_space.n))

    # Init transitions and rewards (ok to loop here)
    # your code here
    

  # the python equivalent for C++ [] operator
  # here we return the policy for the state parameter converted to an integer
  def __getitem__(self, state):
    return self.pi[state].item()

  # reset policy and values
  def reset(self, env):
    self.pi = t.zeros_like(self.pi)
    self.values = t.zeros_like(self.values)

    self.transitions = t.zeros_like(self.transitions)
    self.rewards = t.zeros_like(self.rewards)

    # Re-init transitions and rewards (ok to loop here)
    # your code here
    

  # compute V according to a policy
  def policy_evaluation(self, env, maxit, theta, gamma):
    done = False
    i = 0

    ## you can init things here if needed, for instance:
    transitions = t.zeros((64,64))
    rewards = t.zeros((64,64))

    while not done:
      delta = 0.0
      i += 1

      # your code here
      

      done = (i >= maxit) or (delta < theta)

  # compute a new (better) policy consistent with V
  def policy_improvement(self, env, gamma):
    stable = True

    # your code here
    

    return stable

  # performs the 2-step loop
  # 1. compute V from pi iteratively
  # 2. compute pi from V
  def compute_policy(self, env, max_iteration=1e20, theta=1e-20, gamma=0.9):
    self.reset(env)
    done = False
    i = 0

    while not done:
      i += 1
      self.policy_evaluation(env, max_iteration, theta, gamma)

      done = self.policy_improvement(env, gamma) or i >= max_iteration

In [1]:
env = gym.make('FrozenLake-v1', map_name='8x8', is_slippery=False, render_mode='ansi', new_step_api=True) # is_slippery=False => the lake does not slip, the requested direction is the direction obtained
pi = PolicyIteration(env)
env.reset()

pi.compute_policy(env)
evaluate(env, pi)

print(pi.values.view(8,8))
print(pi.pi.view(8,8))

# If your implementation is successful, you should achieve over 100% success rate and find the shortest path between S and G

In [1]:
env = gym.make('FrozenLake-v1', map_name='8x8', is_slippery=True, render_mode='ansi', new_step_api=True)
pi = PolicyIteration(env)
env.reset()

pi.compute_policy(env)
evaluate(env, pi)

print(pi.values.view(8,8))
print(pi.pi.view(8,8))

# If your implementation is successful, you should achieve over 55% success rate

In [1]:
# To visualize one trial after computing the optimal policy:
env.reset()
evaluate(env, pi, trials=1, render=True)

### But Why?



Your instructor decides to change the discount factor ($\gamma$) to 1 (no devaluation of future rewards).
He relaunches the policy iteration algorithm but the success rate drops from 58% to 0%!

Help your instructor understand why.

*Your answer here&hellip;*



## Value Iteration



Implement the value iteration algorithm as seen in class
Instead of computing state values with the current policy, as is the case with Policy Iteration, compute them with the greedy policy (taking only the action that yields the most).
Only a single policy update is necessary.



In [1]:
class ValueIteration:
  def __init__(self, env):

    self.pi = t.zeros(env.observation_space.n, dtype=t.long)
    self.values = t.zeros(env.observation_space.n)

    self.transitions = t.zeros((env.observation_space.n,env.action_space.n,env.observation_space.n))
    self.rewards = t.zeros((env.observation_space.n,env.action_space.n,env.observation_space.n))

    # same init as before
    # Init transitions and rewards (ok to loop here)
    # your code here
    

  def reset(self, env):
    self.pi = t.zeros_like(self.pi)
    self.values = t.zeros_like(self.values)

    self.transitions = t.zeros_like(self.transitions)
    self.rewards = t.zeros_like(self.rewards)

  def __getitem__(self, state):
    return self.pi[state].item()

  def value_iteration(self, env, maxit, theta, gamma):
    done = False
    i = 0
    while not done:
      delta = 0.0
      i += 1

      # your code here
      
      done = i >= maxit or delta < theta

  def compute_policy(self, env, max_iteration=100000000, theta=1e-20, gamma=0.9):
    self.value_iteration(env, max_iteration, theta, gamma)

    # compute pi from V
    # your code here

In [1]:
env = gym.make('FrozenLake-v1', map_name='8x8', render_mode='ansi', new_step_api=True)
vi = ValueIteration(env)
env.reset()
vi.compute_policy(env)
evaluate(env, vi)

print(pi.values.view(8,8))
print(pi.pi.view(8,8))